# Fashion Guard: Kaggle Inference Lab


Kaggle settings: **Accelerator = GPU or TPU** and in order to first downloading..
**Internet = On**. `ACCELERATOR_MODE` according to your chosing hardware
You made the `auto`, `cuda`, `tpu` or `cpu` . 

`torch_xla` must be find; If the notebook is missing, the system gives the error. You need to work all of nodes step by step. The results writes the `/kaggle/working`.


In [ ]:
import os
from pathlib import Path
import sys

ACCELERATOR_MODE = "auto"
if ACCELERATOR_MODE == "tpu":
    os.environ.setdefault("PJRT_DEVICE", "TPU")

SEARCH_ROOTS = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")] # you can change this part for different environment


def is_repository(path: Path) -> bool:
    return (
        (path / "fashion_guard").is_dir()
        and (path / "benchmark").is_dir()
        and (path / "requirements.txt").is_file()
    )


candidates = []
for search_root in SEARCH_ROOTS:
    if not search_root.exists():
        continue
    candidates.append(search_root)
    candidates.extend(search_root.glob("*"))
    candidates.extend(search_root.glob("*/*"))

REPO_ROOT = next((candidate for candidate in candidates if is_repository(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository bulunamadı. Kaggle Dataset'ini güncelleyip fashion_guard/, benchmark/, "
        "requirements.txt ve kaggle_inference.py dosyalarının olduğu klasörü ekleyin. "
        "benchmark/ klasörü yeni eklendi; Dataset sürümünü yükseltmeyi unutmayın."
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else REPO_ROOT / ".kaggle_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"ACCELERATOR_MODE: {ACCELERATOR_MODE}")


In [ ]:
%pip install -q --upgrade "transformers>=4.53,<5" "accelerate>=0.34" langdetect pandas matplotlib seaborn sentencepiece bitsandbytes
%pip uninstall -y -q torchvision

import importlib.metadata
import importlib.util

if ACCELERATOR_MODE not in {"auto", "cuda", "tpu", "cpu"}:
    raise ValueError("ACCELERATOR_MODE auto, cuda, tpu veya cpu olmalıdır.")

if ACCELERATOR_MODE == "tpu" and importlib.util.find_spec("torch_xla") is None:
    raise RuntimeError(
        "TPU modu torch_xla gerektirir. Kaggle Notebook ayarlarında Accelerator=TPU seçin "
        "ve TPU uyumlu Kaggle image ile yeni bir oturum başlatın. torch/torch_xla paketlerini "
        "bu notebook içinde yükseltmeyin; Kaggle'ın eşleşmiş sürümlerini koruyun."
    )

for package in ("torch", "torch_xla", "transformers", "accelerate", "bitsandbytes", "langdetect", "pandas", "matplotlib", "seaborn"):
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: bulunamadı")

if importlib.util.find_spec("bitsandbytes") is None:
    raise RuntimeError(
        "bitsandbytes kurulu değil; 4-bit quantization için gereklidir. "
        "Bu hücreyi yeniden çalıştırın veya interneti açık tutup tekrar deneyin."
    )

transformers_version = importlib.metadata.version("transformers")
if tuple(int(part) for part in transformers_version.split(".")[:2]) < (4, 53):
    raise RuntimeError(
        f"transformers {transformers_version} kuruldu ama >=4.53 gerekli. "
        "Kaggle'da 'Restart Session' yapıp bu hücreyi yeniden çalıştırın."
    )


In [ ]:
import json
import time

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from fashion_guard.guard import FashionGuard
from fashion_guard.model import QwenGuardModel
from kaggle_inference import run_smoke_tests, runtime_details

print("Fashion Guard import edildi.")
print(json.dumps(runtime_details(), ensure_ascii=False, indent=2))

In [ ]:
import shutil
import sys
import os
from pathlib import Path

# hf_transfer isn't installed in this image; force-disable even if the
# environment already exported HF_HUB_ENABLE_HF_TRANSFER=1.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

if "transformers" in sys.modules:
    raise RuntimeError(
        "transformers bu oturumda zaten import edilmiş. Kaggle'da 'Restart Session' yapıp "
        "hücreleri baştan çalıştırın; aksi halde yükseltilmiş sürüm etkili olmaz."
    )

stale_modules_dir = Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules"
if stale_modules_dir.exists():
    shutil.rmtree(stale_modules_dir)

smoke_results = run_smoke_tests()
print(json.dumps(smoke_results, ensure_ascii=False, indent=2))

MODEL_CANDIDATES = {
    "qwen3_4b": "Qwen/Qwen3-4B-Instruct-2507",
    "qwen3_8b": "Qwen/Qwen3-8B",
    "qwen25_3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen25_7b": "Qwen/Qwen2.5-7B-Instruct",
    "phi4_mini": "microsoft/Phi-4-mini-instruct",
    "gemma2_9b": "google/gemma-2-9b-it",
    "ministral_8b": "mistralai/Ministral-8B-Instruct-2410",
    "granite31_8b": "ibm-granite/granite-3.1-8b-instruct",
    "phi35_mini": "microsoft/Phi-3.5-mini-instruct",
    # A40 üzerinde V4 benchmark'ında test edilen büyük modeller
    "qwen25_14b": "Qwen/Qwen2.5-14B-Instruct",
    "qwen3_14b": "Qwen/Qwen3-14B",
    "mistral_nemo_12b": "mistralai/Mistral-Nemo-Instruct-2407",
    "mistral_small_24b": "mistralai/Mistral-Small-24B-Instruct-2501",
    "qwen3_30b_a3b": "Qwen/Qwen3-30B-A3B-Instruct-2507",
    "granite33_8b": "ibm-granite/granite-3.3-8b-instruct",
    "deepseek_r1_distill_14b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
}

OPTIONAL_GATED_CANDIDATES = {
    "aya_8b_research_only": "CohereLabs/aya-expanse-8b",
    "llama31_8b_gated": "meta-llama/Llama-3.1-8B-Instruct",
}

SELECTED_MODEL = "mistral_small_24b"
MODEL_NAME = MODEL_CANDIDATES[SELECTED_MODEL]
print(f"Seçilen model: {SELECTED_MODEL} -> {MODEL_NAME}")
print(f"Accelerator: {ACCELERATOR_MODE}")

model_adapter = QwenGuardModel(MODEL_NAME, accelerator=ACCELERATOR_MODE, quantization="4bit")
guard = FashionGuard(model=model_adapter)

load_started = time.perf_counter()
model_adapter._load()
print(f"Model yüklendi: {MODEL_NAME}")
print(f"Aktif accelerator: {model_adapter.active_accelerator}")
print(f"Yükleme süresi: {(time.perf_counter() - load_started):.1f} sn")


In [ ]:
GENERATION_CONFIG = {
    "model_name": MODEL_NAME,
    "accelerator_requested": ACCELERATOR_MODE,
    "accelerator_active": model_adapter.active_accelerator,
    "do_sample": False,
    "max_new_tokens": 20,
    "temperature": None,
    "top_p": None,
    "top_k": None,
    "repetition_penalty": None,
}
print(json.dumps(GENERATION_CONFIG, ensure_ascii=False, indent=2))


def route_name(intent: str, route_or_response: str) -> str:
    if route_or_response == "SEARCH":
        return "SEARCH"
    return "NEEDS_MORE_DETAIL" if intent == "nothing_to_search" else "REJECT"


def token_count(text: str) -> int:
    return len(model_adapter.tokenizer(text, add_special_tokens=False)["input_ids"])


def classify_one(
    query: str,
    session_language: str | None = None,
    awaiting_clarification: bool = False,
    context: list[str] | None = None,
) -> dict[str, object]:
    route_or_response, result = guard.route(
        query,
        context=context,
        session_language=session_language,
        awaiting_clarification=awaiting_clarification,
        request_id="kaggle-notebook",
        session_id="manual-session",
    )
    return {
        "query": query,
        "intent": result.intent,
        "route": route_name(result.intent, route_or_response),
        "language": result.language,
        "latency_ms": round(result.latency_ms, 2),
        "input_tokens": token_count(query),
        "output_tokens": token_count(result.raw_output),
        "raw_output": result.raw_output,
        "response": None if route_or_response == "SEARCH" else route_or_response,
        "accelerator": model_adapter.active_accelerator,
    }


In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, display

SUPPORTED_SESSION_LANGUAGES = [
    ("Auto-detect", None),
    ("Turkish (tr-TR)", "tr-TR"),
    ("English (en-GB)", "en-GB"),
    ("German (de-DE)", "de-DE"),
    ("French (fr-FR)", "fr-FR"),
    ("Italian (it-IT)", "it-IT"),
    ("Spanish (es-ES)", "es-ES"),
]

query_input = widgets.Textarea(
    value="Bu elbiseyle hangi ayakkabı gider?",
    placeholder="Test etmek istediğiniz kullanıcı mesajını yazın...",
    description="Sorgu:",
    layout=widgets.Layout(width="100%", height="100px"),
)
language_input = widgets.Dropdown(
    options=SUPPORTED_SESSION_LANGUAGES,
    value=None,
    description="Oturum dili:",
    layout=widgets.Layout(width="360px"),
)
context_input = widgets.Textarea(
    value="",
    placeholder="İsteğe bağlı: önceki user/assistant mesajları, her satır bir mesaj olacak şekilde...",
    description="Bağlam:",
    layout=widgets.Layout(width="100%", height="90px"),
)
clarification_input = widgets.Checkbox(
    value=False,
    description="Bu sorgu önceki clarification sorusuna yanıt",
)
run_button = widgets.Button(
    description="Guard'ı Çalıştır",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="180px", height="40px"),
)
result_output = widgets.Output(layout=widgets.Layout(width="100%"))


def display_manual_result(result: dict[str, object]) -> None:
    route = str(result["route"])
    color = {"SEARCH": "#0b7285", "REJECT": "#c92a2a", "NEEDS_MORE_DETAIL": "#e8590c"}[route]
    user_response = result["response"] or "Downstream agent'a yönlendirilir; guard sabit bir kullanıcı mesajı üretmez."
    rows = [
        ("Intent", result["intent"]),
        ("Route", route),
        ("Dil", result["language"] or "algılanamadı"),
        ("Gecikme", f"{result['latency_ms']} ms"),
        ("Accelerator", result["accelerator"]),
        ("Ham model çıktısı", result["raw_output"] or "Gate tarafından çözüldü"),
        ("Kullanıcıya gidecek çıktı", user_response),
    ]
    table_rows = "".join(
        f"<tr><th style='text-align:left;vertical-align:top;padding:8px;border-bottom:1px solid #ddd'>{name}</th>"
        f"<td style='padding:8px;border-bottom:1px solid #ddd;white-space:pre-wrap'>{value}</td></tr>"
        for name, value in rows
    )
    display(HTML(
        f"<div style='border:1px solid {color};border-radius:6px;padding:12px;margin-top:12px'>"
        f"<div style='font-weight:700;color:{color};font-size:16px'>Guard kararı: {route}</div>"
        f"<table style='width:100%;border-collapse:collapse;margin-top:8px'>{table_rows}</table></div>"
    ))


def run_manual_inference(_button) -> None:
    with result_output:
        result_output.clear_output(wait=True)
        query = query_input.value.strip()
        if not query:
            display(HTML("<b style='color:#c92a2a'>Sorgu boş olamaz.</b>"))
            return
        context = [line.strip() for line in context_input.value.splitlines() if line.strip()][-3:]
        run_button.disabled = True
        try:
            result = classify_one(
                query,
                session_language=language_input.value,
                context=context or None,
                awaiting_clarification=clarification_input.value,
            )
            display_manual_result(result)
        finally:
            run_button.disabled = False


run_button.on_click(run_manual_inference)
controls = widgets.VBox([
    query_input,
    widgets.HBox([language_input, clarification_input]),
    context_input,
    run_button,
    result_output,
])
display(controls)


In [ ]:
chat_history: list[dict[str, object]] = []
chat_user_messages: list[str] = []
chat_awaiting_clarification = False

chat_language = widgets.Dropdown(
    options=SUPPORTED_SESSION_LANGUAGES,
    value=None,
    description="Oturum dili:",
    layout=widgets.Layout(width="360px"),
)
chat_input = widgets.Textarea(
    placeholder="Tester mesajını yazın...",
    description="Mesaj:",
    layout=widgets.Layout(width="100%", height="85px"),
)
chat_send = widgets.Button(
    description="Gönder",
    button_style="primary",
    icon="paper-plane",
    layout=widgets.Layout(width="130px", height="38px"),
)
chat_reset = widgets.Button(
    description="Oturumu Temizle",
    icon="trash",
    layout=widgets.Layout(width="160px", height="38px"),
)
chat_transcript = widgets.Output(layout=widgets.Layout(width="100%"))


def render_chat_history() -> None:
    with chat_transcript:
        chat_transcript.clear_output(wait=True)
        if not chat_history:
            display(HTML("<div style='color:#666;padding:8px'>Henüz mesaj yok.</div>"))
            return
        for entry in chat_history:
            route = str(entry["route"])
            color = {"SEARCH": "#0b7285", "REJECT": "#c92a2a", "NEEDS_MORE_DETAIL": "#e8590c"}[route]
            guard_response = entry["response"] or "Agent'a yönlendirildi; guard kullanıcı mesajı üretmez."
            display(HTML(
                "<div style='border:1px solid #ddd;border-radius:6px;padding:10px;margin:10px 0'>"
                f"<div style='font-weight:600'>Tester: {entry['query']}</div>"
                f"<div style='color:{color};font-weight:700;margin-top:6px'>Guard: {entry['intent']} → {route}</div>"
                f"<div style='margin-top:4px;white-space:pre-wrap'><b>Kullanıcıya çıktı:</b> {guard_response}</div>"
                f"<details style='margin-top:6px'><summary>Teknik ayrıntılar</summary>"
                f"Dil: {entry['language'] or 'algılanamadı'}<br>Gecikme: {entry['latency_ms']} ms<br>"
                f"Ham model çıktısı: <code>{entry['raw_output'] or 'Gate tarafından çözüldü'}</code></details>"
                "</div>"
            ))


def send_chat_message(_button) -> None:
    global chat_awaiting_clarification
    query = chat_input.value.strip()
    if not query:
        return
    chat_send.disabled = True
    try:
        result = classify_one(
            query,
            session_language=chat_language.value,
            context=chat_user_messages[-3:] or None,
            awaiting_clarification=chat_awaiting_clarification,
        )
        chat_history.append(result)
        chat_user_messages.append(query)
        chat_user_messages[:] = chat_user_messages[-3:]
        chat_awaiting_clarification = result["intent"] == "nothing_to_search"
        chat_input.value = ""
        render_chat_history()
    finally:
        chat_send.disabled = False


def reset_chat_session(_button) -> None:
    global chat_awaiting_clarification
    chat_history.clear()
    chat_user_messages.clear()
    chat_awaiting_clarification = False
    chat_input.value = ""
    render_chat_history()


chat_send.on_click(send_chat_message)
chat_reset.on_click(reset_chat_session)
chat_panel = widgets.VBox([
    chat_language,
    chat_input,
    widgets.HBox([chat_send, chat_reset]),
    chat_transcript,
])
display(chat_panel)
render_chat_history()


In [ ]:
print(f"Accelerator: {model_adapter.active_accelerator} | device: {model_adapter.device} | dtype: {next(model_adapter.model.parameters()).dtype}")

raw_model_output = model_adapter.generate(MANUAL_QUERY, MANUAL_CONTEXT or None)
print("Ham model çıktısı:")
print(raw_model_output)


In [ ]:
from benchmark.cases import build_cases, write_cases

BENCHMARK_CASE_OBJECTS = build_cases()
BENCHMARK_CASES = [case.__dict__ for case in BENCHMARK_CASE_OBJECTS]
cases_frame = pd.DataFrame(BENCHMARK_CASES)

CASES_PATH = OUTPUT_DIR / "benchmark_v3_cases.jsonl"
write_cases(CASES_PATH, BENCHMARK_CASE_OBJECTS)

runnable_cases = cases_frame[cases_frame["query"].notna()]
print(f"Vakalar yazıldı: {CASES_PATH}")
print(f"Toplam vaka: {len(cases_frame)}")
print(f"Çalıştırılabilir ve otomatik skorlanabilir vaka: {len(runnable_cases)}")
display(cases_frame.groupby(["language", "category"]).size().unstack(fill_value=0))
display(cases_frame["expected_resolution_path"].value_counts().to_frame("cases"))


In [ ]:
from benchmark.runner import run_benchmark, write_results

benchmark_records = run_benchmark(guard, SELECTED_MODEL, MODEL_NAME, cases=BENCHMARK_CASES)
results = pd.DataFrame(benchmark_records)

RESULTS_JSONL_PATH = OUTPUT_DIR / f"benchmark_v3_results_{SELECTED_MODEL}.jsonl"
with RESULTS_JSONL_PATH.open("w", encoding="utf-8") as handle:
    for record in benchmark_records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

RESULTS_CSV_PATH = OUTPUT_DIR / f"benchmark_v3_results_{SELECTED_MODEL}.csv"
write_results(RESULTS_CSV_PATH, benchmark_records)

print(f"Çalıştırılan vaka: {len(benchmark_records)}")
print(f"JSONL: {RESULTS_JSONL_PATH}")
print(f"CSV:   {RESULTS_CSV_PATH}")
results[["case_id", "language", "category", "expected_intent", "intent", "expected_route", "route", "expected_resolution_path", "latency_ms", "raw_output"]]


In [ ]:
from benchmark.evaluate import run as evaluate_run

summary = evaluate_run(CASES_PATH, RESULTS_CSV_PATH)

SUMMARY_PATH = OUTPUT_DIR / f"benchmark_v3_summary_{SELECTED_MODEL}.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Özet: {SUMMARY_PATH}")

if summary["is_invalid_run"]:
    print(f"UYARI: bu çalıştırma GEÇERSİZ sayılıyor (model_error oranı %{summary['model_error_rate']}). Sonuçları başlık tablosuna eklemeyin.")

print(f"Skorlanan vaka: {summary['scored_cases']} | Eksik sonuç: {summary['missing_results']}")
print(f"Gate ile çözülen: {summary['gate_resolved_cases']} (route doğruluğu %{summary['gate_route_accuracy']})")
print(f"Model ile çözülen: {summary['model_resolved_cases']} (route doğruluğu %{summary['model_route_accuracy']}, intent doğruluğu %{summary['model_intent_accuracy']})")
print(f"Genel route doğruluğu: %{summary['overall_route_accuracy']} | Genel intent doğruluğu: %{summary['overall_intent_accuracy']}")
print(f"Beklenmeyen SEARCH: {summary['unexpected_search']} | Model hata oranı: %{summary['model_error_rate']}")

print("\nDil bazında intent doğruluğu (yalnızca modelin gerçekten çözdüğü vakalar):")
display(pd.DataFrame(summary["per_language_intent_accuracy"]).T)

print("\nKategori bazında route doğruluğu:")
display(pd.DataFrame(summary["per_category_route_accuracy"]).T)

print("\nConfusion matrix (yalnızca modelin gerçekten çözdüğü, hatasız vakalar):")
display(pd.Series(summary["confusion_matrix"]).sort_values(ascending=False).to_frame("count"))


In [ ]:
scored_results = results[results["expected_route"] != ""].copy()
scored_results["route_match"] = scored_results["route"] == scored_results["expected_route"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

language_accuracy = scored_results.groupby("language", as_index=False)["route_match"].mean()
sns.barplot(data=language_accuracy, x="language", y="route_match", ax=axes[0], color="#0b7285")
axes[0].set(title="Dil Bazında Route Eşleşmesi", xlabel="Dil", ylabel="Eşleşme oranı")
axes[0].set_ylim(0, 1)
sns.barplot(data=scored_results, x="difficulty", y="latency_ms", ax=axes[1], color="#e8590c")
axes[1].set(title="Zorluk Bazında Ortalama Gecikme", xlabel="Zorluk", ylabel="Latency (ms)")

plt.tight_layout()
plt.show()

display(pd.crosstab(scored_results["expected_intent"], scored_results["intent"], margins=True))
